# Setup Google Colab - Analisis Strategi Penjualan UMKM

Jalankan sel dari atas ke bawah setiap kali membuka sesi Colab baru.

**Persiapan sekali saja:** unggah `umkm_proyek.zip` ke folder utama Google Drive (`MyDrive`).
Sel kedua akan mengekstraknya otomatis ke `MyDrive/umkm_proyek/`. Jika Anda sudah mengunggah
foldernya langsung (bukan zip), sel itu cukup memakai folder tersebut.

Data kuesioner memuat nama usaha: simpan hanya di Drive pribadi, jangan dibagikan publik.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
import zipfile

DRIVE = "/content/drive/MyDrive"
PROJECT_DIR = f"{DRIVE}/umkm_proyek"
ZIP_PATH = f"{DRIVE}/umkm_proyek.zip"

if not os.path.isdir(PROJECT_DIR):
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(f"Unggah dulu umkm_proyek.zip ke {DRIVE}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DRIVE)
    print("Zip diekstrak ke", PROJECT_DIR)

# Semua path paket `umkm` mengikuti UMKM_BASE_DIR (lihat src/umkm/config.py).
os.environ["UMKM_BASE_DIR"] = PROJECT_DIR
sys.path.insert(0, f"{PROJECT_DIR}/src")
os.chdir(PROJECT_DIR)
print("Folder kerja:", os.getcwd())

In [ ]:
# Tahap 1 hanya butuh pustaka yang sudah ada di Colab; baris ini sekadar memastikan.
# Pustaka analisis tahap berikutnya (statsmodels, shap, pingouin) ditambahkan saat dibutuhkan.
!pip install -q pyyaml openpyxl

import pandas as pd

print("pandas", pd.__version__)

## Tes unit (opsional)

In [ ]:
!PYTHONPATH=src python -m pytest -q -p no:cacheprovider tests

## Tahap 1: Cleaning

Membaca `data/raw/Kuesioner_Penelitian__Responses_.xlsx`, lalu menulis
`data/processed/data_clean.csv` dan `reports/cleaning_log.md` di Drive.

In [ ]:
from IPython.display import Markdown, display

from umkm import cleaning, config

df, rep = cleaning.run()
print(f"data_clean.csv: {df.shape[0]} baris x {df.shape[1]} kolom")
display(Markdown((config.path("reports") / "cleaning_log.md").read_text(encoding="utf-8")))

In [ ]:
df.drop(columns="nama_usaha").head()

## Tahap 2: Konstruksi variabel

Membaca `data_clean.csv`, lalu menulis `data/processed/data_features.csv`, tabel di
`reports/tables/`, dan `reports/features_log.md`.

In [ ]:
from umkm import features

feat, frep = features.run()
print(f"data_features.csv: {feat.shape[0]} baris x {feat.shape[1]} kolom")
display(Markdown((config.path("reports") / "features_log.md").read_text(encoding="utf-8")))

## Tahap 3: EDA

Membaca `data_features.csv` dan `data_clean.csv`, lalu menulis `reports/tables/eda_*.csv`,
`reports/figures/eda_*.png`, `reports/eda_log.md`, dan `reports/eda_ringkasan.md` di Drive.
Butuh scipy, statsmodels, dan matplotlib (sudah tersedia di Colab).

In [ ]:
from IPython.display import Image

from umkm import eda

erep = eda.run()
print(f"{len(erep.tabel)} tabel, {len(erep.grafik)} grafik")
display(Markdown((config.path("reports") / "eda_ringkasan.md").read_text(encoding="utf-8")))

In [ ]:
for path in erep.grafik.values():
    display(Image(filename=str(path)))

## Tahap 4: Analisis pengaruh

Membaca `data_features.csv`, lalu menulis `reports/tables/pengaruh_*.csv`, grafik `pengaruh_*.png`
dan `shap_*.png`, `reports/pengaruh_log.md`, dan `reports/pengaruh_ringkasan.md` di Drive.
Butuh `shap` (dipasang di sel ini). Bootstrap 1000x memakan beberapa menit.

In [ ]:
!pip install -q shap

import warnings

from umkm import pengaruh

warnings.filterwarnings("ignore")
prep = pengaruh.run()
print(f"{len(prep.tabel)} tabel, {len(prep.grafik)} grafik, bootstrap {prep.info['n_boot']}x")
display(Markdown((config.path("reports") / "pengaruh_ringkasan.md").read_text(encoding="utf-8")))

In [ ]:
for path in prep.grafik.values():
    display(Image(filename=str(path)))

## Tahap 5: Segmentasi

Membaca `data_features.csv`, lalu menulis `data/processed/data_segmen.csv`,
`reports/tables/profil_segmen.csv` dan `segmen_*.csv`, grafik `segmen_*.png`,
`reports/segmentasi_log.md`, dan `reports/segmentasi_ringkasan.md` di Drive.

In [ ]:
from umkm import segmentation

srep = segmentation.run()
print(f"k = {srep.info['k']}, silhouette {srep.info['silhouette']:.3f}")
display(Markdown((config.path("reports") / "segmentasi_ringkasan.md").read_text(encoding="utf-8")))

In [ ]:
for path in srep.grafik.values():
    display(Image(filename=str(path)))